In [1]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import numpy as np
import nilearn.image
import nilearn.plotting
import tempfile
import copy
import shutil
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

In [2]:
dcm_root = Path("/Users/william.wakefield/PycharmProjects/ADRU/model_data/adni/t1_long_data/t1_long_dcm")
out_dir = Path("/Users/william.wakefield/PycharmProjects/ADRU/model_data/adni/t1_long_data/t1_long_raw_v2")

scan_dirs = sorted(p for p in dcm_root.glob("*/*/*/*") if p.is_dir())
print(f"Found {len(scan_dirs)} scan directories")

Found 802 scan directories


In [3]:
failed = []
skipped = 0
for scan_dir in tqdm(scan_dirs, desc="Converting DCM to NII"):
    image_id = scan_dir.name
    subject_id = scan_dir.parts[-4]
    out_name = f"{image_id}_{subject_id}"

    if (out_dir / f"{out_name}.nii").exists():
        skipped += 1
        continue

    result = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(out_dir), "-b", "n", str(scan_dir)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        failed.append((out_name, result.stderr.strip()))
        
nii_files = list(out_dir.glob("*.nii"))
print(f"\nConversion complete: {len(nii_files)} .nii files ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} conversions failed:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Converting DCM to NII: 100%|██████████| 802/802 [00:00<00:00, 113616.10it/s]


Conversion complete: 802 .nii files (802 skipped, already existed)


In [ ]:
mni_t1 = ants.image_read('model_data/mni_t1.nii')

In [ ]:
raw_dir = Path("model_data/adni/t1_long_data/t1_long_raw")
reg_dir = Path("model_data/adni/t1_long_data/t1_long_registered_v2")

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Found {len(raw_files)} raw T1 images to register")

Found 802 raw T1 images to register


In [ ]:
failed = []
skipped = 0

for raw_path in tqdm(raw_files, desc="Registering to MNI"):
    stem = raw_path.stem
    out_nii = reg_dir / f"{stem}.nii"
    out_affine = reg_dir / f"{stem}_0GenericAffine.mat"
    out_warp = reg_dir / f"{stem}_1Warp.nii.gz"
    out_inv_warp = reg_dir / f"{stem}_1InverseWarp.nii.gz"

    if out_nii.exists() and out_affine.exists() and out_warp.exists():
        skipped += 1
        continue

    try:
        moving = ants.image_read(str(raw_path))
        reg = ants.registration(fixed=mni_t1, moving=moving, type_of_transform="SyN")

        ants.image_write(reg["warpedmovout"], str(out_nii))

        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)

        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv_warp)

    except Exception as e:
        failed.append((stem, str(e)))

registered = list(reg_dir.glob("*.nii"))
transforms = list(reg_dir.glob("*.mat"))
print(f"\nRegistration complete: {len(registered)} registered images, {len(transforms)} transforms")
print(f"({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} registrations failed:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Registering to MNI: 100%|██████████| 802/802 [44:37<00:00,  3.34s/it]


Registration complete: 802 registered images, 802 transforms
(0 skipped, already existed)


In [2]:
reg_dir = Path("model_data/adni/t1_long_data/t1_long_registered")
strip_dir = Path("model_data/adni/t1_long_data/t1_long_skull_strip")

reg_files = sorted(reg_dir.glob("*.nii"))
print(f"Found {len(reg_files)} registered T1 images to skull-strip")

failed = []
skipped = 0

Found 802 registered T1 images to skull-strip


In [ ]:
for reg_path in tqdm(reg_files, desc="Skull stripping (BET)"):
    stem = reg_path.stem
    out_nii = strip_dir / f"{stem}.nii"

    if out_nii.exists():
        skipped += 1
        continue

    bet_gz = strip_dir / f"{stem}_brain.nii.gz"
    try:
        result = subprocess.run(
            ["bet", str(reg_path), str(bet_gz), "-R"],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            failed.append((stem, result.stderr.strip()))
            continue

        img = nib.load(str(bet_gz))
        nib.save(img, str(out_nii))
        bet_gz.unlink()

    except Exception as e:
        failed.append((stem, str(e)))

stripped = list(strip_dir.glob("*.nii"))
print(f"\nSkull stripping complete: {len(stripped)} .nii files")
print(f"({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

In [5]:
# Run FSL FAST segmentation on all skull-stripped T1 images
strip_dir = Path("model_data/adni/t1_long_data/t1_long_skull_strip")
seg_output_dir = Path("model_data/adni/t1_long_data/t1_long_seg_output")

strip_files = sorted(strip_dir.glob("*.nii"))
print(f"Found {len(strip_files)} skull-stripped T1 images to segment")

Found 802 skull-stripped T1 images to segment


In [6]:
failed = []
skipped = 0

for input_t1_path in tqdm(strip_files, desc="FAST segmentation"):
    stem = input_t1_path.stem
    image_output_dir = seg_output_dir / stem
    image_output_dir.mkdir(parents=True, exist_ok=True)

    out_base = image_output_dir / stem
    seg_file = image_output_dir / f"{stem}_seg.nii"
    if seg_file.exists():
        skipped += 1
        continue

    result = subprocess.run(
        ["fast", "-o", str(out_base), str(input_t1_path)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        failed.append((stem, result.stderr.strip()))

seg_dirs = [p for p in seg_output_dir.iterdir() if p.is_dir()]
print(f"\nFAST segmentation complete: {len(seg_dirs)} output folders")
print(f"({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

FAST segmentation: 100%|██████████| 802/802 [11:27:03<00:00, 51.40s/it]    


FAST segmentation complete: 802 output folders
(0 skipped, already existed)


In [7]:
dti_dcm_root = Path("model_data/adni/dti_long_data/dti_long_dcm")
dti_out_dir = Path("model_data/adni/dti_long_data/dti_long_registered_2")

dti_scan_dirs = sorted(p for p in dti_dcm_root.glob("*/*/*/*") if p.is_dir())
print(f"Found {len(dti_scan_dirs)} DTI scan directories")

Found 802 DTI scan directories


In [8]:
failed = []
skipped = 0

for scan_dir in tqdm(dti_scan_dirs, desc="Converting DTI DCM to NII"):
    image_id = scan_dir.name
    subject_id = scan_dir.parts[-4]
    out_name = f"{image_id}_{subject_id}"

    if (dti_out_dir / f"{out_name}.nii").exists():
        skipped += 1
        continue

    result = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(dti_out_dir), "-b", "n", str(scan_dir)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        failed.append((out_name, result.stderr.strip()))

nii_files = list(dti_out_dir.glob("*.nii"))
print(f"\nDTI conversion complete: {len(nii_files)} .nii files ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} conversions failed:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Converting DTI DCM to NII: 100%|██████████| 802/802 [14:35<00:00,  1.09s/it]  


DTI conversion complete: 836 .nii files (0 skipped, already existed)


In [14]:
import re

dti_raw_dir = Path("model_data/adni/dti_long_data/dti_long_raw")
dti_strip_dir = Path("model_data/adni/dti_long_data/dti_long_skull_strip")

primary_pattern = re.compile(r"^I\d+_\d{3}_S_\d+\.nii$")
dti_raw_files = sorted(f for f in dti_raw_dir.glob("*.nii") if primary_pattern.match(f.name))
print(f"Found {len(dti_raw_files)} primary DTI images to skull-strip")

Found 802 primary DTI images to skull-strip


In [15]:
failed = []
skipped = 0

for dti_path in tqdm(dti_raw_files, desc="DTI skull stripping (BET)"):
    stem = dti_path.stem
    out_nii = dti_strip_dir / f"{stem}.nii.gz"
    mask_path = dti_strip_dir / f"{stem}_mask.nii.gz"
    bval_src = dti_raw_dir / f"{stem}.bval"
    bvec_src = dti_raw_dir / f"{stem}.bvec"
    bval_dst = dti_strip_dir / f"{stem}.bval"
    bvec_dst = dti_strip_dir / f"{stem}.bvec"

    if out_nii.exists() and mask_path.exists() and bval_dst.exists() and bvec_dst.exists():
        skipped += 1
        continue

    b0_path = dti_strip_dir / f"{stem}_b0.nii.gz"
    b0_brain_base = dti_strip_dir / f"{stem}_b0_brain"
    b0_brain = dti_strip_dir / f"{stem}_b0_brain.nii.gz"
    b0_mask = dti_strip_dir / f"{stem}_b0_brain_mask.nii.gz"

    try:
        subprocess.run(
            ["fslroi", str(dti_path), str(b0_path), "0", "1"],
            capture_output=True, text=True, check=True,
        )

        subprocess.run(
            ["bet", str(b0_path), str(b0_brain_base), "-m", "-f", "0.3"],
            capture_output=True, text=True, check=True,
        )

        subprocess.run(
            ["fslmaths", str(dti_path), "-mas", str(b0_mask), str(out_nii)],
            capture_output=True, text=True, check=True,
        )

        shutil.move(str(b0_mask), str(mask_path))

        if bval_src.exists():
            shutil.copy(bval_src, bval_dst)
        if bvec_src.exists():
            shutil.copy(bvec_src, bvec_dst)

        for tmp in [b0_path, b0_brain]:
            if tmp.exists():
                tmp.unlink()

    except subprocess.CalledProcessError as e:
        failed.append((stem, e.stderr.strip()))
    except Exception as e:
        failed.append((stem, str(e)))

stripped = list(dti_strip_dir.glob("I*_[0-9]*_S_[0-9]*.nii.gz"))
masks = list(dti_strip_dir.glob("*_mask.nii.gz"))
bvals = list(dti_strip_dir.glob("*.bval"))
bvecs = list(dti_strip_dir.glob("*.bvec"))
print(f"\nDTI skull stripping complete:")
print(f"  {len(stripped)} skull-stripped .nii.gz files")
print(f"  {len(masks)} brain masks")
print(f"  {len(bvals)} .bval files")
print(f"  {len(bvecs)} .bvec files")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

DTI skull stripping (BET): 100%|██████████| 802/802 [57:51<00:00,  4.33s/it]  


DTI skull stripping complete:
  1604 skull-stripped .nii.gz files
  802 brain masks
  801 .bval files
  801 .bvec files
  (0 skipped, already existed)


In [17]:
dti_strip_dir = Path("model_data/adni/dti_long_data/dti_long_skull_strip")
dtifit_dir = Path("model_data/adni/dti_long_data/dti_long_ditfit")

dtifit_primary = re.compile(r"^I\d+_\d{3}_S_\d+\.nii\.gz$")
dti_files = sorted(f for f in dti_strip_dir.glob("*.nii.gz") if dtifit_primary.match(f.name))
print(f"Found {len(dti_files)} skull-stripped DTI images to fit")

Found 802 skull-stripped DTI images to fit


In [18]:
failed = []
skipped = 0

for dti_path in tqdm(dti_files, desc="DTIFIT"):
    stem = dti_path.name.replace(".nii.gz", "")
    mask_path = dti_strip_dir / f"{stem}_mask.nii.gz"
    bval_path = dti_strip_dir / f"{stem}.bval"
    bvec_path = dti_strip_dir / f"{stem}.bvec"

    subject_dir = dtifit_dir / stem
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_base = subject_dir / stem
    fa_file = subject_dir / f"{stem}_FA.nii.gz"

    if fa_file.exists():
        skipped += 1
        continue

    missing = [p for p in [mask_path, bval_path, bvec_path] if not p.exists()]
    if missing:
        failed.append((stem, f"missing inputs: {[p.name for p in missing]}"))
        continue

    result = subprocess.run(
        [
            "dtifit",
            "-k", str(dti_path),
            "-m", str(mask_path),
            "-r", str(bvec_path),
            "-b", str(bval_path),
            "-o", str(out_base),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        failed.append((stem, result.stderr.strip()))

fa_maps = list(dtifit_dir.glob("*/*_FA.nii.gz"))
md_maps = list(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"\nDTIFIT complete:")
print(f"  {len(fa_maps)} FA maps")
print(f"  {len(md_maps)} MD maps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

DTIFIT: 100%|██████████| 802/802 [23:09<00:00,  1.73s/it]


DTIFIT complete:
  801 FA maps
  801 MD maps
  (0 skipped, already existed)
1 failures:
  I1612816_128_S_0205: missing inputs: ['I1612816_128_S_0205.bval', 'I1612816_128_S_0205.bvec']


In [19]:
dtifit_dir = Path("model_data/adni/dti_long_data/dti_long_ditfit")
t1_raw_dir = Path("model_data/adni/t1_long_data/t1_long_raw")
t1_reg_dir = Path("model_data/adni/t1_long_data/t1_long_registered")
mni_t1_path = Path("model_data/mni_t1.nii")

md_to_t1_dir = Path("model_data/adni/dti_long_data/dti_long_reg_t1")
md_to_mni_dir = Path("model_data/adni/dti_long_data/dti_long_mni")

subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")

md_maps = sorted(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"Found {len(md_maps)} DTI MD maps to register")

Found 801 DTI MD maps to register


In [20]:
t1_by_subj = {}
for f in sorted(t1_raw_dir.glob("I*_*_S_*.nii")):
    m = subj_pat.match(f.stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(f.stem)

dti_by_subj = {}
for md_path in md_maps:
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    m = subj_pat.match(dti_stem)
    if m:
        dti_by_subj.setdefault(m.group(1), []).append((dti_stem, md_path))

dti_to_t1 = {}
for subj, dti_list in dti_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    dti_list_sorted = sorted(dti_list, key=lambda x: x[0])
    for i, (dti_stem, _) in enumerate(dti_list_sorted):
        if not t1_list:
            continue
        t1_stem = t1_list[i] if i < len(t1_list) else t1_list[-1]
        dti_to_t1[dti_stem] = t1_stem

mni_t1 = ants.image_read(str(mni_t1_path))

In [21]:
failed = []
skipped = 0

for md_path in tqdm(md_maps, desc="Registering MD → T1 → MNI"):
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(dti_stem)

    if t1_stem is None:
        failed.append((dti_stem, "no matching T1 for subject"))
        continue

    md_in_t1 = md_to_t1_dir / f"{dti_stem}_MD.nii.gz"
    md_rigid_tx = md_to_t1_dir / f"{dti_stem}_rigid.mat"
    md_in_mni = md_to_mni_dir / f"{dti_stem}_MD.nii.gz"

    if md_in_t1.exists() and md_rigid_tx.exists() and md_in_mni.exists():
        skipped += 1
        continue

    t1_raw_path = t1_raw_dir / f"{t1_stem}.nii"
    t1_affine = t1_reg_dir / f"{t1_stem}_0GenericAffine.mat"
    t1_warp = t1_reg_dir / f"{t1_stem}_1Warp.nii.gz"

    missing = [p for p in [t1_raw_path, t1_affine, t1_warp] if not p.exists()]
    if missing:
        failed.append((dti_stem, f"missing: {[p.name for p in missing]}"))
        continue

    try:
        fixed_t1 = ants.image_read(str(t1_raw_path))
        moving_md = ants.image_read(str(md_path))

        rigid_reg = ants.registration(
            fixed=fixed_t1, moving=moving_md, type_of_transform="Rigid"
        )
        ants.image_write(rigid_reg["warpedmovout"], str(md_in_t1))

        for tx_path in rigid_reg["fwdtransforms"]:
            if "GenericAffine" in Path(tx_path).name or tx_path.endswith(".mat"):
                shutil.copy(tx_path, md_rigid_tx)
                break

        md_mni = ants.apply_transforms(
            fixed=mni_t1,
            moving=rigid_reg["warpedmovout"],
            transformlist=[str(t1_warp), str(t1_affine)],
            interpolator="linear",
        )
        ants.image_write(md_mni, str(md_in_mni))

    except Exception as e:
        failed.append((dti_stem, str(e)))

in_t1 = list(md_to_t1_dir.glob("*_MD.nii.gz"))
in_mni = list(md_to_mni_dir.glob("*_MD.nii.gz"))
print(f"\nRegistration complete:")
print(f"  {len(in_t1)} MD maps in T1 space ({md_to_t1_dir})")
print(f"  {len(in_mni)} MD maps in MNI space ({md_to_mni_dir})")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Registering MD → T1 → MNI:  16%|█▋        | 131/801 [05:19<20:38,  1.85s/it]Exception Object caught: 

itk::ExceptionObject (0x176bc1320)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x13709c090): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.


Registering MD → T1 → MNI:  28%|██▊       | 224/801 [09:07<16:54,  1.76s/it]Exception Object caught: 

itk::ExceptionObject (0x31ab90cd0)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x31ab96440): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.


Registering MD → T1 → MNI:  31%|███       | 246/801 [10:02<18:20,  1.98s/it]Exception Object caught:


Registration complete:
  787 MD maps in T1 space (model_data/adni/dti_long_data/dti_long_reg_t1)
  787 MD maps in MNI space (model_data/adni/dti_long_data/dti_long_mni)
  (0 skipped, already existed)
14 failures:
  I1092244_114_S_6039: Registration failed with error code 1
  I1224877_020_S_6513: Registration failed with error code 1
  I1241880_020_S_5203: Registration failed with error code 1
  I1278648_020_S_6185: Registration failed with error code 1
  I1325865_020_S_6449: Registration failed with error code 1
  I1329976_020_S_6513: Registration failed with error code 1
  I1332415_020_S_6227: Registration failed with error code 1
  I1349126_020_S_6504: Registration failed with error code 1
  I1412031_020_S_6901: Registration failed with error code 1
  I1428968_022_S_5004: Registration failed with error code 1


In [22]:
md_mni_dir = Path("model_data/adni/dti_long_data/dti_long_mni")
seg_dir = Path("model_data/adni/dti_long_data/dti_long_segment_output")

md_files = sorted(md_mni_dir.glob("*_MD.nii.gz"))
print(f"Found {len(md_files)} MD maps to segment")

Found 787 MD maps to segment


In [23]:
failed = []
skipped = 0

for md_path in tqdm(md_files, desc="FAST segmentation"):
    stem = md_path.name.replace("_MD.nii.gz", "")

    subject_dir = seg_dir / stem
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_base = subject_dir / stem

    pve_csf = subject_dir / f"{stem}_pve_0.nii.gz"
    pve_gm = subject_dir / f"{stem}_pve_1.nii.gz"
    pve_wm = subject_dir / f"{stem}_pve_2.nii.gz"

    if pve_csf.exists() and pve_gm.exists() and pve_wm.exists():
        skipped += 1
        continue

    result = subprocess.run(
        [
            "fast",
            "-t", "2",
            "-n", "3",
            "-o", str(out_base),
            str(md_path),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        failed.append((stem, result.stderr.strip()))

csf_maps = list(seg_dir.glob("*/*_pve_0.nii.gz"))
gm_maps = list(seg_dir.glob("*/*_pve_1.nii.gz"))
wm_maps = list(seg_dir.glob("*/*_pve_2.nii.gz"))
print(f"\nFAST segmentation complete:")
print(f"  {len(csf_maps)} CSF maps (pve_0)")
print(f"  {len(gm_maps)} GM maps (pve_1)")
print(f"  {len(wm_maps)} WM maps (pve_2)")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

FAST segmentation:   4%|▎         | 29/787 [12:10<5:18:08, 25.18s/it]


KeyboardInterrupt: 